# Federal Funds Target Rate - FOMC Meeting Data Processing

**Goal:**
1. Process the `Federal Funds Target Range Upper Limit.csv` data.
2. Incorporate explicit FOMC meeting dates to identify decision days.
3. Create labels (`higher`, `same`, `lower`) reflecting the decision made at each meeting compared to the previous meeting.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the continuous daily data
file_path = 'Federal Funds Target Range Upper Limit.csv'
df = pd.read_csv(file_path)

# Clean Data
df['DFEDTARU'] = pd.to_numeric(df['DFEDTARU'], errors='coerce')
df['observation_date'] = pd.to_datetime(df['observation_date'])
df = df.dropna(subset=['DFEDTARU']).sort_values('observation_date').reset_index(drop=True)

df.head()

### Incorporate FOMC Meeting Dates

We use the provided list of past FOMC meeting dates. The new target rate takes effect on the meeting day itself or the day after. Because the FRED series is continuous, measuring the target rate exactly **1 day after** the meeting will securely yield the post-decision target rate for that meeting.

In [ ]:
raw_meetings = [
    '2009-01-28', '2009-03-18', '2009-04-29', '2009-06-24', '2009-08-12', '2009-09-23', '2009-11-04', '2009-12-16', 
    '2010-01-27', '2010-03-16', '2010-04-28', '2010-06-23', '2010-08-10', '2010-09-21', '2010-11-03', '2010-12-14', 
    '2011-01-26', '2011-03-15', '2011-04-27', '2011-06-22', '2011-08-09', '2011-09-21', '2011-11-02', '2011-12-13', 
    '2012-01-25', '2012-03-13', '2012-04-25', '2012-06-20', '2012-07-31', '2012-09-13', '2012-10-24', '2012-12-12', 
    '2013-01-30', '2013-03-20', '2013-05-01', '2013-06-19', '2013-07-31', '2013-09-18', '2013-10-30', '2013-12-18', 
    '2014-01-29', '2014-03-19', '2014-04-30', '2014-06-18', '2014-07-30', '2014-09-17', '2014-10-29', '2014-12-17', 
    '2015-01-28', '2015-03-18', '2015-04-29', '2015-06-17', '2015-07-29', '2015-09-17', '2015-10-28', '2015-12-16', 
    '2016-01-27', '2016-03-16', '2016-04-27', '2016-06-15', '2016-07-27', '2016-09-21', '2016-11-02', '2016-12-14', 
    '2017-02-01', '2017-03-15', '2017-05-03', '2017-06-14', '2017-07-26', '2017-09-20', '2017-11-01', '2017-12-13', 
    '2018-01-31', '2018-03-21', '2018-05-02', '2018-06-13', '2018-08-01', '2018-09-26', '2018-11-08', '2018-12-19', 
    '2019-01-30', '2019-03-19', '2019-05-01', '2019-06-19', '2019-07-31', '2019-09-18', '2019-10-04', '2019-10-30', '2019-12-11', 
    '2020-01-29', '2020-03-03', '2020-03-15', '2020-03-23', '2020-04-29', '2020-06-10', '2020-07-29', '2020-09-16', '2020-11-05', '2020-12-16', 
    '2021-01-27', '2021-03-17', '2021-04-28', '2021-06-16', '2021-07-28', '2021-09-22', '2021-11-03', '2021-12-15', 
    '2022-01-26', '2022-03-16', '2022-05-04', '2022-06-15', '2022-07-27', '2022-09-21', '2022-11-02', '2022-12-14', 
    '2023-02-01', '2023-03-22', '2023-05-03', '2023-06-14', '2023-07-26', '2023-09-20', '2023-11-01', '2023-12-13', 
    '2024-01-31', '2024-03-20', '2024-05-01', '2024-06-12', '2024-07-31', '2024-09-18', '2024-11-07', '2024-12-18', 
    '2025-01-28', '2025-03-19', '2025-05-07', '2025-06-18', '2025-07-30', '2025-09-17', '2025-10-29', '2025-12-10', 
    '2026-01-28'
]

# Ensure date formats are correct and drop any accidental duplicates (like 2015-04-29)
df_meetings = pd.DataFrame({'meeting_date': pd.to_datetime(raw_meetings)})
df_meetings = df_meetings.drop_duplicates().sort_values('meeting_date').reset_index(drop=True)

# The decision establishes the target rate that holds the day after the meeting
df_meetings['post_meeting_date'] = df_meetings['meeting_date'] + pd.Timedelta(days=1)

# Merge with the continuous target rate dataframe
df_decisions = pd.merge(
    df_meetings,
    df[['observation_date', 'DFEDTARU']],
    left_on='post_meeting_date',
    right_on='observation_date',
    how='left'
)

# Clean up columns and keep only relevant variables
df_decisions = df_decisions[['meeting_date', 'DFEDTARU']].rename(columns={'DFEDTARU': 'target_rate'})

print("Total meetings matched to rates:", len(df_decisions))
df_decisions.head()

### Determine FOMC Action Labels (`higher`, `same`, `lower`)

Now we calculate the difference between the target rate produced by the *current* meeting and the target rate maintained from the *previous* meeting.

In [ ]:
def get_action_label(diff_val):
    if pd.isna(diff_val):
        return None
    elif diff_val > 0:
        return 'higher'
    elif diff_val < 0:
        return 'lower'
    else:
        return 'same'

# Calculate diff from the immediately preceding meeting
df_decisions['previous_rate'] = df_decisions['target_rate'].shift(1)
df_decisions['rate_change'] = df_decisions['target_rate'] - df_decisions['previous_rate']
df_decisions['decision'] = df_decisions['rate_change'].apply(get_action_label)

display(df_decisions.head(15))

### Explore Rate Policy Distribution

Let's see an overview of what the Federal Reserve actually did across these meetings.

In [ ]:
print("Overall Rate Changes at Meetings (Distribution):")
display(df_decisions['decision'].value_counts())

print("\nSample of Meetings where rates CHANGED:")
display(df_decisions[df_decisions['decision'].isin(['higher', 'lower'])].head(10))